# Lean-31 : les opérateurs de Hecke $T_p$ et $U_p$ — compagnon natif du lake `hecke_lean`

**Série** : [SymbolicAI / Lean](README.md) — companions natifs (cf. [Lean-26 calibration](Lean-26-Calibration-Native-Companion.ipynb), [Lean-24b ERC-20](Lean-24b-Lean-ERC20-Native-Companion.ipynb))

Compagnon **natif** du lake [`hecke_lean`](hecke_lean/) : ici le lake est **importé et exécuté** dans un kernel Lean 4 réel (`lean4-wsl`), et chaque définition / théorème est interrogé par `#check` ou `#print axioms` — les sorties de ce notebook sont des sorties du compilateur Lean, pas de la prose à propos de Lean.

Le lake formalise les **opérateurs de Hecke classiques** sur le demi-plan supérieur $\mathbb{H}$ : pour un entier $p$ (usuellement premier), l'opérateur $T_p$ agit sur une fonction $f : \mathbb{H} \to \mathbb{C}$ par une somme finie de slashs sur des représentants explicites des classes $\Gamma(1) \backslash M_2(\mathbb{Z})$ de déterminant $p$, et l'opérateur $U_p$ n'en retient que la partie triangulaire. La formule induite sur les coefficients de Fourier — $a(np) + p^{k-1}\,a(n/p)$ selon que $p$ divise $n$ ou non — est portée par `coeffHeckeT`.

**Provenance** : le lake est un port du dépôt `anthropics/fermats-last-theorem` (fichier `Definitions/Def_ModularForm_HeckeOperator.lean`, commit `aa2d8b34692b`), docstrings pédagogiques en français et exemples calculables ajoutés ; licence Apache-2.0 préservée (voir [`hecke_lean/NOTICE.md`](hecke_lean/NOTICE.md)).

## Pourquoi ce compagnon

Un notebook « natif » ne **décrit** pas le lake : il le **charge**. Chaque `#check` ci-dessous est résolu par le compilateur Lean contre les `.olean` compilés du lake — la signature affichée est celle qui vit dans `Hecke/HeckeOperator.lean`, pas une copie. La contrepartie pédagogique est précieuse pour la théorie des formes modulaires : les opérateurs de Hecke y sont souvent présentés comme des boîtes noires algébriques ; ici, chaque brique (représentants, slash, déterminants, coefficients) est interrogée séparément et son énoncé exact est affiché.

## Plan du notebook

1. **Le lake** : architecture, provenance, conventions du kernel
2. **Les représentants** $\gamma_{p,j}$ et la partie diagonale
3. **L'action sur $\mathbb{H}$** : slash, dénominateurs, homothéties
4. **Les opérateurs** $U_p$ et $T_p$ : définitions, lectures ponctuelles, linéarité
5. **La formule des coefficients** : `coeffHeckeT` et ses exemples calculables
6. **Transparence axiomatique et limites**

## Conventions du notebook

- **Kernel** : `lean4-wsl` (Lean 4 via WSL, exécuté depuis le répertoire du lake)
- **Imports** : en tête de la première cellule code uniquement — le kernel partage un seul environnement entre cellules, où `import` n'est légal qu'en tête de session, comme en tête de fichier Lean (le chargement de la fermeture Mathlib prend de l'ordre de la minute)
- **Sorties** : `#check` type, `#print axioms` vérifie la preuve, `#eval` calcule
- **Exercices** : convention C.1 — le notebook s'exécute de bout en bout ; chaque cellule d'exercice énonce l'objectif en commentaire, le laisse ouvert sous un unique `-- TODO étudiant` (non résolu) et rappelle le lemme utile par un `#check` — aucun script de solution n'est fourni

### Substance formelle

| Notion | Symbole du lake | Énoncé clé |
|--------|-----------------|------------|
| Représentant triangulaire | `heckeMatrix p j` | $\gamma_{p,j} = \begin{pmatrix} 1 & j \\ 0 & p \end{pmatrix}$, $\det = p$ |
| Représentant diagonal | `heckeDiagMatrix p` | $\begin{pmatrix} p & 0 \\ 0 & 1 \end{pmatrix}$, $\det = p$ |
| Partie triangulaire | `heckeU k p f` | $\sum_{j<p} f \mid [k]\ \gamma_{p,j}$ |
| Opérateur de Hecke | `heckeT k p f` | $U_p f + f \mid [k]\ \text{diag}$ |
| Coefficients | `coeffHeckeT k p a n` | $a(np) + p^{k-1} a(n/p)$ si $p \mid n$ |

### Prérequis

- [Lean-5 Tactics](Lean-5-Tactics.ipynb) (`simp`, `rw`, `decide`) et [Lean-6 Mathlib](Lean-6-Mathlib-Essentials.ipynb)
- Théorie : action de $SL_2(\mathbb{Z})$ sur $\mathbb{H}$ par homographies ; le notebook rappelle ce qu'il utilise

### Durée estimée

35 à 45 minutes en lecture interactive (kernel WSL requis pour l'exécution réelle).

Les sections 4 et 5 sont le cœur : on y voit la **géométrie** (le slash) devenir **combinatoire** (les coefficients de Fourier).

## 1. Le lake `hecke_lean` : un port pédagogique, trois familles d'énoncés

`hecke_lean` est un lake mono-module : `Hecke/HeckeOperator.lean` (et son miroir i18n `HeckeOperator_en.lean`, convention sibling pair de l'EPIC #4980), adossé à Mathlib via `Mathlib.NumberTheory.ModularForms.SlashActions`, pinné au commit `db584cd6d46c` sur la toolchain Lean `v4.33.0`. Le fichier racine `Hecke.lean` n'est qu'un agrégateur d'import.

Le module organise ses énoncés en trois familles :

1. **Les représentants** : `upperTriangularGL`, `heckeMatrix`, `heckeDiagMatrix` — la géométrie des classes $\Gamma(1) \backslash M_2(\mathbb{Z})$ de déterminant $p$ ;
2. **Les opérateurs** : `heckeU`, `heckeT` et leurs lectures ponctuelles — l'analyse sur $\mathbb{H}$ ;
3. **Les coefficients** : `coeffHeckeT`, `coeffHeckeU` — la combinatoire des suites de Fourier, avec une section `Examples` d'exemples calculables absents du dépôt amont.

Le produit de Petersson et les cusp forms sont **hors périmètre** de cette première tranche (grain aval).

**Pourquoi la toolchain compte ici** : le kernel `lean4-wsl` lance le REPL Lean avec le `LEAN_PATH` du lake — les `.olean` ne se chargent que si la version du compilateur qui les a produits correspond à celle du REPL. C'est la raison pour laquelle ce notebook s'exécute depuis le répertoire du lake : c'est là que le kernel détecte le workspace et sa toolchain.

In [1]:
-- TOUTES les importations de la session viennent ici (tête de session) :
-- le kernel lean4-wsl partage UN environnement entre cellules, où `import`
-- n'est légal qu'en tête de session, comme en tête de fichier Lean.
import Hecke.HeckeOperator

-- Les notations du lake (GL, matrices !![...]) puis son namespace :
open scoped MatrixGroups
open ModularForm

#check @ModularForm.heckeMatrix        -- le représentant γ_{p,j} = !![1, j; 0, p]
#check @ModularForm.heckeDiagMatrix    -- le représentant diagonal !![p, 0; 0, 1]
#check @ModularForm.heckeU             -- la partie triangulaire U_p
#check @ModularForm.heckeT             -- l'opérateur de Hecke T_p = U_p + diagonal
#check @ModularForm.coeffHeckeT        -- la formule des coefficients de T_p
#check @ModularForm.coeffHeckeU        -- l'échantillonnage a (n p) de U_p

-- TOUTES les importations de la session viennent ici (tête de session) :
-- le kernel lean4-wsl partage UN environnement entre cellules, où `import`
-- n'est légal qu'en tête de session, comme en tête de fichier Lean.
import Hecke.HeckeOperator

-- Les notations du lake (GL, matrices !![...]) puis son namespace :
open scoped MatrixGroups
open ModularForm

#check @ModularForm.heckeMatrix        -- le représentant γ_{p,j} = !![1, j; 0, p]
──────▶  heckeMatrix : ℕ → ℕ → GL (Fin 2) ℝ
#check @ModularForm.heckeDiagMatrix    -- le représentant diagonal !![p, 0; 0, 1]
──────▶  heckeDiagMatrix : ℕ → GL (Fin 2) ℝ
#check @ModularForm.heckeU             -- la partie triangulaire U_p
──────▶  heckeU : ℤ → ℕ → (UpperHalfPlane → ℂ) → UpperHalfPlane → ℂ
#check @ModularForm.heckeT             -- l'opérateur de Hecke T_p = U_p + diagonal
──────▶  heckeT : ℤ → ℕ → (UpperHalfPlane → ℂ) → UpperHalfPlane → ℂ
#check @ModularForm.coeffHeckeT        -- la formule des coefficients de T_p
──────▶  coeffHeckeT : ℤ → ℕ → (ℕ → ℂ) → ℕ → ℂ
#check @ModularForm.coeffHeckeU        -- l'échantillonnage a (n p) de U_p
──────▶  coeffHeckeU : ℕ → (ℕ → ℂ) → ℕ → ℂ
--% env 0

Raw input:
{"cmd": "-- TOUTES les importations de la session viennent ici (t\u00eate de session) :\n-- le kernel lean4-wsl partage UN environnement entre cellules, o\u00f9 `import`\n-- n'est l\u00e9gal qu'en t\u00eate de session, comme en t\u00eate de fichier Lean.\nimport Hecke.HeckeOperator\n\n-- Les notations du lake (GL, matrices !![...]) puis son namespace :\nopen scoped MatrixGroups\nopen ModularForm\n\n#check @ModularForm.heckeMatrix        -- le repr\u00e9sentant \u03b3_{p,j} = !![1, j; 0, p]\n#check @ModularForm.heckeDiagMatrix    -- le repr\u00e9sentant diagonal !![p, 0; 0, 1]\n#check @ModularForm.heckeU             -- la partie triangulaire U_p\n#check @ModularForm.heckeT             -- l'op\u00e9rateur de Hecke T_p = U_p + diagonal\n#check @ModularForm.coeffHeckeT        -- la formule des coefficients de T_p\n#check @ModularForm.coeffHeckeU        -- l'\u00e9chantillonnage a (n p) de U_p"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data": "heckeMatrix : ℕ → ℕ → GL (Fin 2) ℝ"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data": "heckeDiagMatrix : ℕ → GL (Fin 2) ℝ"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 6},
   "data": "heckeU : ℤ → ℕ → (UpperHalfPlane → ℂ) → UpperHalfPlane → ℂ"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 6},
   "data": "heckeT : ℤ → ℕ → (UpperHalfPlane → ℂ) → UpperHalfPlane → ℂ"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 6},
   "data": "coeffHeckeT : ℤ → ℕ → (ℕ → ℂ) → ℕ → ℂ"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 6},
   "data": "coeffHeckeU : ℕ → (ℕ → ℂ) → ℕ → ℂ"}],
 "env": 0}

### Lecture des signatures

**Sortie obtenue** : six signatures, rendues par le compilateur contre les oleans du lake.

| Symbole | Signature | Rôle |
|---------|-----------|------|
| `heckeMatrix` | `ℕ → ℕ → GL (Fin 2) ℝ` | le représentant triangulaire $\gamma_{p,j}$ |
| `heckeDiagMatrix` | `ℕ → GL (Fin 2) ℝ` | le représentant diagonal, même déterminant |
| `heckeU` | `ℤ → ℕ → (ℍ → ℂ) → ℍ → ℂ` | somme des $p$ slashs triangulaires |
| `heckeT` | `ℤ → ℕ → (ℍ → ℂ) → ℍ → ℂ` | $T_p = U_p + $ slash diagonal |
| `coeffHeckeT` | `ℤ → ℕ → (ℕ → ℂ) → ℕ → ℂ` | la suite de Fourier de $T_p f$ |
| `coeffHeckeU` | `ℕ → (ℕ → ℂ) → ℕ → ℂ` | l'échantillonnage $a \mapsto a (n\,p)$ |

**Points clés** :

1. Les opérateurs prennent le **poids** $k : \mathbb{Z}$ en premier argument — c'est lui qui portera le facteur $p^{k-1}$ ;
2. Ils agissent sur des fonctions **arbitraires** $\mathbb{H} \to \mathbb{C}$ : aucune modularité n'est exigée à ce stade, $T_p$ est d'abord un endomorphisme d'un espace de fonctions ;
3. Les opérateurs à coefficients (`coeffHeckeT`) vivent sur les **suites** $a : \mathbb{N} \to \mathbb{C}$ — la traduction combinatoire de l'action géométrique.

> **Note technique** : les deux niveaux (fonctions sur $\mathbb{H}$ / suites de Fourier) coexistent dans le même module sans être reliés par un théorème de passage — ce pont (développement en $q$-série) appartient au grain aval du lake.

## 2. Les représentants $\gamma_{p,j}$ et la partie diagonale

L'opérateur $T_p$ se définit par une somme sur les classes à gauche $\Gamma(1) \backslash \{ M \in M_2(\mathbb{Z}) : \det M = p \}$. Pour $p$ premier, cette orbite admet $p+1$ représentants : les $p$ matrices triangulaires $\gamma_{p,j} = \begin{pmatrix} 1 & j \\ 0 & p \end{pmatrix}$ pour $j = 0, \dots, p-1$ (la partie « $U$ »), plus la matrice diagonale $\begin{pmatrix} p & 0 \\ 0 & 1 \end{pmatrix}$.

Le lake encode la brique commune par `upperTriangularGL a b d` : la matrice $\begin{pmatrix} a & b \\ 0 & d \end{pmatrix}$ vue dans $GL(2, \mathbb{R})$, avec l'hypothèse `a * d ≠ 0` qui garantit l'inversibilité. Les deux familles de représentants en sont des instances :

- `heckeMatrix p j := upperTriangularGL 1 j p` (si $p = 0$, renvoie l'identité — cas dégénéré neutralisé) ;
- `heckeDiagMatrix p := upperTriangularGL p 0 1`.

Les théorèmes `val_heckeMatrix` et `val_heckeDiagMatrix` (`@[simp]`) donnent les **valeurs** explicites ; `det_heckeMatrix` et `det_heckeDiagMatrix` assurent que le déterminant vaut **exactement** $p$ — et `det_heckeMatrix_pos` qu'il est positif, la condition qui garantit que les représentants préservent $\mathbb{H}$.

In [2]:
-- Les briques : la matrice triangulaire et ses valeurs explicites.
#check @ModularForm.upperTriangularGL
#check @ModularForm.val_heckeMatrix
#check @ModularForm.val_heckeDiagMatrix

-- Le déterminant des représentants vaut EXACTEMENT p (pas |p|) :
#check @ModularForm.det_heckeMatrix
#check @ModularForm.det_heckeDiagMatrix
#check @ModularForm.det_heckeMatrix_pos

-- Certificat : preuve close, sans axiome au-delà des standards.
#print axioms ModularForm.det_heckeMatrix

-- Les briques : la matrice triangulaire et ses valeurs explicites.
#check @ModularForm.upperTriangularGL
──────▶  upperTriangularGL : (a : ℝ) → ℝ → (d : ℝ) → a * d ≠ 0 → GL (Fin 2) ℝ
#check @ModularForm.val_heckeMatrix
──────▶  @val_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ), ↑(heckeMatrix p j) = !![1, ↑j; 0, ↑p]
#check @ModularForm.val_heckeDiagMatrix
──────▶  @val_heckeDiagMatrix : ∀ {p : ℕ}, p ≠ 0 → ↑(heckeDiagMatrix p) = !![↑p, 0; 0, 1]

-- Le déterminant des représentants vaut EXACTEMENT p (pas |p|) :
#check @ModularForm.det_heckeMatrix
──────▶  @det_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ), ↑(Matrix.GeneralLinearGroup.det (heckeMatrix p j)) = ↑p
#check @ModularForm.det_heckeDiagMatrix
──────▶  @det_heckeDiagMatrix : ∀ {p : ℕ}, p ≠ 0 → ↑(Matrix.GeneralLinearGroup.det (heckeDiagMatrix p)) = ↑p
#check @ModularForm.det_heckeMatrix_pos
──────▶  det_heckeMatrix_pos : ∀ (p j : ℕ), 0 < ↑(Matrix.GeneralLinearGroup.det (heckeMatrix p j))

-- Certificat : preuve close, sans axiome au-delà des standards.
#print axioms ModularForm.det_heckeMatrix
──────▶  'ModularForm.det_heckeMatrix' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 1

Raw input:
{"cmd": "-- Les briques : la matrice triangulaire et ses valeurs explicites.\n#check @ModularForm.upperTriangularGL\n#check @ModularForm.val_heckeMatrix\n#check @ModularForm.val_heckeDiagMatrix\n\n-- Le d\u00e9terminant des repr\u00e9sentants vaut EXACTEMENT p (pas |p|) :\n#check @ModularForm.det_heckeMatrix\n#check @ModularForm.det_heckeDiagMatrix\n#check @ModularForm.det_heckeMatrix_pos\n\n-- Certificat : preuve close, sans axiome au-del\u00e0 des standards.\n#print axioms ModularForm.det_heckeMatrix", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "upperTriangularGL : (a : ℝ) → ℝ → (d : ℝ) → a * d ≠ 0 → GL (Fin 2) ℝ"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@val_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ), ↑(heckeMatrix p j) = !![1, ↑j; 0, ↑p]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@val_heckeDiagMatrix : ∀ {p : ℕ}, p ≠ 0 → ↑(heckeDiagMatrix p) = !![↑p, 0; 0, 1]"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "@det_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ), ↑(Matrix.GeneralLinearGroup.det (heckeMatrix p j)) = ↑p"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@det_heckeDiagMatrix : ∀ {p : ℕ}, p ≠ 0 → ↑(Matrix.GeneralLinearGroup.det (heckeDiagMatrix p)) = ↑p"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "det_heckeMatrix_pos : ∀ (p j : ℕ), 0 < ↑(Matrix.GeneralLinearGroup.det (heckeMatrix p j))"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 6},
   "data":
   "'ModularForm.det_heckeMatrix' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 1}

### Lecture : valeurs et déterminants

**Sortie obtenue** : les équations de valeur (`val_heckeMatrix hp j : heckeMatrix p j = !![1, j; 0, p]`), les déterminants (`((heckeMatrix p j).det : ℝ) = p`), et le certificat d'axiomes de `det_heckeMatrix`.

| Énoncé | Ce qu'il dit |
|--------|--------------|
| `val_heckeMatrix` | $\gamma_{p,j}$ est **littéralement** la matrice $\begin{pmatrix} 1 & j \\ 0 & p \end{pmatrix}$ à coefficients réels |
| `det_heckeMatrix` | son déterminant vaut $p$ — l'indice de l'opérateur, pas une valeur absolue |
| `det_heckeMatrix_pos` | déterminant **positif** y compris pour $p = 0$ (identité) : l'action préserve $\mathbb{H}$ |

**Points clés** :

1. Le déterminant est énoncé dans $\mathbb{R}$ via le coercion `((...).det : ℝ) = p` — le $p$ de droite est un naturel coercé, l'égalité est exacte ;
2. Pour `det_heckeMatrix`, `#print axioms` ne liste que les axiomes standards (`propext`, `Classical.choice`, `Quot.sound` selon la sortie ci-dessus) et aucun `sorryAx` : ce certificat établit que **ce théorème interrogé** a une preuve close. La propriété globale « zéro `sorry` » repose séparément sur le scan du source du lake.

> **Note technique** : la positivité du déterminant n'est pas un détail : c'est elle qui distingue les représentants de Hecke des éléments de $GL_2^-(\mathbb{R})$, qui échangeraient les deux demi-plans.

### Exercice 1 — lire la valeur d'un représentant

Sur le modèle de `val_heckeMatrix`, établissez la **valeur explicite** du représentant $\gamma_{3,2} = \begin{pmatrix} 1 & 2 \\ 0 & 3 \end{pmatrix}$ : l'énoncé compare la coercion de `heckeMatrix 3 2` dans `Matrix (Fin 2) (Fin 2) ℝ` au littéral `!![1, 2; 0, 3]`.

**Indice** : `val_heckeMatrix` prend l'hypothèse `p ≠ 0` — pour $p = 3$, elle se décharge par `decide` ; `rw` ramène ensuite le but à une égalité de littéraux numériques, que `rfl` referme (explicite : le `rfl` implicite de `rw` ne déplie pas assez les coercitions `↑2` contre `2`).

In [3]:
-- Exercice 1 : la valeur explicite du représentant γ_{3,2} (déterminant 3).
--
-- Objectif : établir
--   ((ModularForm.heckeMatrix 3 2 : GL (Fin 2) ℝ) : Matrix (Fin 2) (Fin 2) ℝ)
--       = !![(1 : ℝ), 2; 0, 3]
-- TODO étudiant : à compléter (indice dans la cellule précédente).

#check @ModularForm.val_heckeMatrix   -- le lemme à invoquer

-- Exercice 1 : la valeur explicite du représentant γ_{3,2} (déterminant 3).
--
-- Objectif : établir
--   ((ModularForm.heckeMatrix 3 2 : GL (Fin 2) ℝ) : Matrix (Fin 2) (Fin 2) ℝ)
--       = !![(1 : ℝ), 2; 0, 3]
-- TODO étudiant : à compléter (indice dans la cellule précédente).

#check @ModularForm.val_heckeMatrix   -- le lemme à invoquer
──────▶  @val_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ), ↑(heckeMatrix p j) = !![1, ↑j; 0, ↑p]
--% env 2

Raw input:
{"cmd": "-- Exercice 1 : la valeur explicite du repr\u00e9sentant \u03b3_{3,2} (d\u00e9terminant 3).\n--\n-- Objectif : \u00e9tablir\n--   ((ModularForm.heckeMatrix 3 2 : GL (Fin 2) \u211d) : Matrix (Fin 2) (Fin 2) \u211d)\n--       = !![(1 : \u211d), 2; 0, 3]\n-- TODO \u00e9tudiant : \u00e0 compl\u00e9ter (indice dans la cellule pr\u00e9c\u00e9dente).\n\n#check @ModularForm.val_heckeMatrix   -- le lemme \u00e0 invoquer", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@val_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ), ↑(heckeMatrix p j) = !![1, ↑j; 0, ↑p]"}],
 "env": 2}

## 3. L'action sur le demi-plan supérieur : le slash

Le groupe $GL_2^+(\mathbb{R})$ agit sur $\tau \in \mathbb{H}$ par homographies, et cette action se relève en l'**action de slash** de poids $k$ sur les fonctions :

$$(f \mid [k]\ \gamma)(\tau) = \det(\gamma)^{k-1} \cdot \sigma(\gamma)\big(c\tau + d\big)^{-k} \cdot f(\gamma \cdot \tau)$$

Le lake décompose cette formule pour chacun des deux représentants :

- pour $\gamma_{p,j}$ : le dénominateur $c\tau + d$ vaut $p$ (théorème `denom_heckeMatrix`), le caractère $\sigma$ est trivial (déterminant positif), et l'action sur $\tau$ est l'homothétie-translation $(\tau + j)/p$ (`coe_heckeMatrix_smul`) — d'où la lecture $(f \mid [k]\ \gamma_{p,j})(\tau) = p^{-1} f\big((\tau + j)/p\big)$ ;
- pour le diagonal : le dénominateur vaut $1$, l'action est la dilatation $p\,\tau$, et le déterminant $p$ porte le facteur $p^{k-1}$ — d'où $(f \mid [k]\ \mathrm{diag})(\tau) = p^{k-1} f(p\,\tau)$.

Les deux théorèmes `slash_heckeMatrix_apply` et `slash_heckeDiagMatrix_apply` sont ces lectures **démontrées** — ce sont eux qui feront le pont entre la définition abstraite de $T_p$ et la formule des coefficients.

In [4]:
-- L'action des représentants sur τ ∈ ℍ :
#check @ModularForm.coe_heckeMatrix_smul       -- γ_{p,j} • τ = (τ + j) / p
#check @ModularForm.coe_heckeDiagMatrix_smul   -- diag • τ = p • τ
#check @ModularForm.denom_heckeMatrix          -- dénominateur p
#check @ModularForm.denom_heckeDiagMatrix      -- dénominateur 1
#check @ModularForm.σ_heckeMatrix              -- caractère σ trivial (det > 0)

-- Les lectures du slash sur chaque famille :
#check @ModularForm.slash_heckeMatrix_apply
#check @ModularForm.slash_heckeDiagMatrix_apply

-- L'action des représentants sur τ ∈ ℍ :
#check @ModularForm.coe_heckeMatrix_smul       -- γ_{p,j} • τ = (τ + j) / p
──────▶  @coe_heckeMatrix_smul : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ) (τ : UpperHalfPlane), ↑(heckeMatrix p j • τ) = (↑τ + ↑j) / ↑p
#check @ModularForm.coe_heckeDiagMatrix_smul   -- diag • τ = p • τ
──────▶  @coe_heckeDiagMatrix_smul : ∀ {p : ℕ}, p ≠ 0 → ∀ (τ : UpperHalfPlane), ↑(heckeDiagMatrix p • τ) = ↑p * ↑τ
#check @ModularForm.denom_heckeMatrix          -- dénominateur p
──────▶  @denom_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ) (τ : UpperHalfPlane), UpperHalfPlane.denom (heckeMatrix p j) ↑τ = ↑p
#check @ModularForm.denom_heckeDiagMatrix      -- dénominateur 1
──────▶  @denom_heckeDiagMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (τ : UpperHalfPlane), UpperHalfPlane.denom (heckeDiagMatrix p) ↑τ = 1
#check @ModularForm.σ_heckeMatrix              -- caractère σ trivial (det > 0)
──────▶  σ_heckeMatrix : ∀ (p j : ℕ), UpperHalfPlane.σ (heckeMatrix p j) = ContinuousAlgEquiv.refl ℝ ℂ

-- Les lectures du slash sur chaque famille :
#check @ModularForm.slash_heckeMatrix_apply
──────▶  slash_heckeMatrix_apply : ∀ (k : ℤ) {p : ℕ},
  p ≠ 0 →
    ∀ (j : ℕ) (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),
      (f ∣[k] heckeMatrix p j) τ = (↑p)⁻¹ * f (heckeMatrix p j • τ)
#check @ModularForm.slash_heckeDiagMatrix_apply
──────▶  slash_heckeDiagMatrix_apply : ∀ (k : ℤ) {p : ℕ},
  p ≠ 0 →
    ∀ (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),
      (f ∣[k] heckeDiagMatrix p) τ = ↑p ^ (k - 1) * f (heckeDiagMatrix p • τ)
--% env 3

Raw input:
{"cmd": "-- L'action des repr\u00e9sentants sur \u03c4 \u2208 \u210d :\n#check @ModularForm.coe_heckeMatrix_smul       -- \u03b3_{p,j} \u2022 \u03c4 = (\u03c4 + j) / p\n#check @ModularForm.coe_heckeDiagMatrix_smul   -- diag \u2022 \u03c4 = p \u2022 \u03c4\n#check @ModularForm.denom_heckeMatrix          -- d\u00e9nominateur p\n#check @ModularForm.denom_heckeDiagMatrix      -- d\u00e9nominateur 1\n#check @ModularForm.\u03c3_heckeMatrix              -- caract\u00e8re \u03c3 trivial (det > 0)\n\n-- Les lectures du slash sur chaque famille :\n#check @ModularForm.slash_heckeMatrix_apply\n#check @ModularForm.slash_heckeDiagMatrix_apply", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@coe_heckeMatrix_smul : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ) (τ : UpperHalfPlane), ↑(heckeMatrix p j • τ) = (↑τ + ↑j) / ↑p"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@coe_heckeDiagMatrix_smul : ∀ {p : ℕ}, p ≠ 0 → ∀ (τ : UpperHalfPlane), ↑(heckeDiagMatrix p • τ) = ↑p * ↑τ"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@denom_heckeMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (j : ℕ) (τ : UpperHalfPlane), UpperHalfPlane.denom (heckeMatrix p j) ↑τ = ↑p"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "@denom_heckeDiagMatrix : ∀ {p : ℕ}, p ≠ 0 → ∀ (τ : UpperHalfPlane), UpperHalfPlane.denom (heckeDiagMatrix p) ↑τ = 1"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "σ_heckeMatrix : ∀ (p j : ℕ), UpperHalfPlane.σ (heckeMatrix p j) = ContinuousAlgEquiv.refl ℝ ℂ"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "slash_heckeMatrix_apply : ∀ (k : ℤ) {p : ℕ},\n  p ≠ 0 →\n    ∀ (j : ℕ) (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),\n      (f ∣[k] heckeMatrix p j) τ = (↑p)⁻¹ * f (heckeMatrix p j • τ)"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "slash_heckeDiagMatrix_apply : ∀ (k : ℤ) {p : ℕ},\n  p ≠ 0 →\n    ∀ (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),\n      (f ∣[k] heckeDiagMatrix p) τ = ↑p ^ (k - 1) * f (heckeDiagMatrix p • τ)"}],
 "

### Lecture : deux comportements opposés du slash

**Sortie obtenue** : les homothéties $(\gamma_{p,j} \bullet \tau : \mathbb{C}) = (\tau + j)/p$ et $(\mathrm{diag} \bullet \tau : \mathbb{C}) = p \cdot \tau$, plus les deux lectures du slash.

| Représentant | Action sur $\tau$ | Slash de poids $k$ en $\tau$ |
|--------------|--------------------|-------------------------------|
| $\gamma_{p,j}$ (triangulaire) | $(\tau + j)/p$ — $p$ translatées **écrasées** vers la pointe | $p^{-1} f\big((\tau+j)/p\big)$ |
| diagonal | $p\,\tau$ — **dilatation** vers l'intérieur | $p^{k-1} f(p\,\tau)$ |

**Points clés** :

1. La partie $U$ **contracte** le voisinage de la pointe ($\tau \mapsto (\tau+j)/p$ envoie $\mathbb{H}$ proche de $0$), tandis que le diagonal **l'étire** ($\tau \mapsto p\tau$) — les deux morceaux de $T_p$ explorent des régions opposées du demi-plan ;
2. Le facteur de poids se loge entièrement dans le terme diagonal ($p^{k-1}$) : la partie triangulaire ne porte que $p^{-1}$, indépendant de $k$ ;
3. Ces lectures sont les **seuls endroits** du module où la formule générale du slash est effectivement calculée — en aval (`heckeU_apply`, `heckeT_apply`), tout s'exprime à partir d'elles.

> **Note technique** : $\sigma$ trivial (`σ_heckeMatrix`) signifie pas de conjugaison supplémentaire — c'est une conséquence directe de la positivité du déterminant vue en section 2.

## 4. Les opérateurs $U_p$ et $T_p$

Les définitions tombent maintenant naturellement :

$$U_p f = \sum_{j=0}^{p-1} f \mid [k]\ \gamma_{p,j}, \qquad T_p f = U_p f + f \mid [k]\ \begin{pmatrix} p & 0 \\ 0 & 1 \end{pmatrix}$$

Le lake en donne trois niveaux de lecture : la définition (`heckeU_def`, `heckeT_def`, par sommes finies sur `Finset.range p`), la **lecture ponctuelle** (`heckeU_apply`, `heckeT_apply` — la valeur en un $\tau$ fixé) et le cas dégénéré `heckeT_zero_left` : pour $p = 0$, $T_0 = \mathrm{id}$. La linéarité en $f$ (`heckeT_add`, `heckeT_smul`, et leurs versions $U$) fait de chaque $T_p$ un **endomorphisme** de l'espace des fonctions $\mathbb{H} \to \mathbb{C}$ — le décor minimal pour une théorie spectrale des formes modulaires.

In [5]:
-- Les opérateurs, par leurs définitions et lectures ponctuelles :
#check @ModularForm.heckeU_def
#check @ModularForm.heckeT_def
#check @ModularForm.heckeU_apply
#check @ModularForm.heckeT_apply

-- Cas dégénéré p = 0 : T₀ est l'identité (lemme @[simp]).
#check @ModularForm.heckeT_zero_left

-- Certificat : la lecture ponctuelle de T_p est une preuve close.
#print axioms ModularForm.heckeT_apply

-- Les opérateurs, par leurs définitions et lectures ponctuelles :
#check @ModularForm.heckeU_def
──────▶  heckeU_def : ∀ (k : ℤ) (p : ℕ) (f : UpperHalfPlane → ℂ), heckeU k p f = ∑ j ∈ Finset.range p, f ∣[k] heckeMatrix p j
#check @ModularForm.heckeT_def
──────▶  heckeT_def : ∀ (k : ℤ) (p : ℕ) (f : UpperHalfPlane → ℂ),
  heckeT k p f = ∑ j ∈ Finset.range p, f ∣[k] heckeMatrix p j + f ∣[k] heckeDiagMatrix p
#check @ModularForm.heckeU_apply
──────▶  heckeU_apply : ∀ (k : ℤ) {p : ℕ},
  p ≠ 0 →
    ∀ (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),
      heckeU k p f τ = (↑p)⁻¹ * ∑ j ∈ Finset.range p, f (heckeMatrix p j • τ)
#check @ModularForm.heckeT_apply
──────▶  heckeT_apply : ∀ (k : ℤ) {p : ℕ},
  p ≠ 0 →
    ∀ (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),
      heckeT k p f τ = (↑p)⁻¹ * ∑ j ∈ Finset.range p, f (heckeMatrix p j • τ) + ↑p ^ (k - 1) * f (heckeDiagMatrix p • τ)

-- Cas dégénéré p = 0 : T₀ est l'identité (lemme @[simp]).
#check @ModularForm.heckeT_zero_left
──────▶  heckeT_zero_left : ∀ (k : ℤ) (f : UpperHalfPlane → ℂ), heckeT k 0 f = f

-- Certificat : la lecture ponctuelle de T_p est une preuve close.
#print axioms ModularForm.heckeT_apply
──────▶  'ModularForm.heckeT_apply' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 4

Raw input:
{"cmd": "-- Les op\u00e9rateurs, par leurs d\u00e9finitions et lectures ponctuelles :\n#check @ModularForm.heckeU_def\n#check @ModularForm.heckeT_def\n#check @ModularForm.heckeU_apply\n#check @ModularForm.heckeT_apply\n\n-- Cas d\u00e9g\u00e9n\u00e9r\u00e9 p = 0 : T\u2080 est l'identit\u00e9 (lemme @[simp]).\n#check @ModularForm.heckeT_zero_left\n\n-- Certificat : la lecture ponctuelle de T_p est une preuve close.\n#print axioms ModularForm.heckeT_apply", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "heckeU_def : ∀ (k : ℤ) (p : ℕ) (f : UpperHalfPlane → ℂ), heckeU k p f = ∑ j ∈ Finset.range p, f ∣[k] heckeMatrix p j"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "heckeT_def : ∀ (k : ℤ) (p : ℕ) (f : UpperHalfPlane → ℂ),\n  heckeT k p f = ∑ j ∈ Finset.range p, f ∣[k] heckeMatrix p j + f ∣[k] heckeDiagMatrix p"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "heckeU_apply : ∀ (k : ℤ) {p : ℕ},\n  p ≠ 0 →\n    ∀ (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),\n      heckeU k p f τ = (↑p)⁻¹ * ∑ j ∈ Finset.range p, f (heckeMatrix p j • τ)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "heckeT_apply : ∀ (k : ℤ) {p : ℕ},\n  p ≠ 0 →\n    ∀ (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),\n      heckeT k p f τ = (↑p)⁻¹ * ∑ j ∈ Finset.range p, f (heckeMatrix p j • τ) + ↑p ^ (k - 1) * f (heckeDiagMatrix p • τ)"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "heckeT_zero_left : ∀ (k : ℤ) (f : UpperHalfPlane → ℂ), heckeT k 0 f = f"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "'ModularForm.heckeT_apply' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 4}

### Lecture : la formule ponctuelle de $T_p$

**Sortie obtenue** : les définitions par sommes finies, et surtout `heckeT_apply` :

$$\big(T_p f\big)(\tau) = \frac{1}{p} \sum_{j=0}^{p-1} f\!\left(\frac{\tau + j}{p}\right) + p^{k-1} f(p\,\tau)$$

| Terme | Origine géométrique |
|-------|---------------------|
| $p^{-1} \sum_j f\big((\tau+j)/p\big)$ | la partie $U$ : $p$ translatées écrasées, facteur $1/p$ du dénominateur |
| $p^{k-1} f(p\tau)$ | le diagonal : dilatation, facteur de poids du déterminant |

**Points clés** :

1. Cette formule est **démontrée**, pas posée : `heckeT_apply` la déduit de la définition par sommes + les lectures du slash de la section 3 — et `#print axioms` atteste que la preuve est close ;
2. `heckeT_zero_left` fait de $p = 0$ un cas **exact** ($T_0 f = f$, refermé dans le lake par `simp [heckeT]` : la somme vide et le slash par l'identité restituent $f$) — la définition est robuste au cas dégénéré sans clause ad hoc ;
3. On voit ici la parenté avec l'opérateur $U_p$ des formes à niveau $p$ : `heckeU_apply` isole le premier terme — c'est l'opérateur qui, appliqué à une $q$-série, **garde** les coefficients d'indice multiple de $p$.

### Exercice 2 — le cas dégénéré $T_0 = \mathrm{id}$

Prouvez en **une seule tactique** que $T_0$ est l'identité : l'énoncé `heckeT k 0 f = f` est exactement le lemme `heckeT_zero_left` du lake, déjà enregistré comme `@[simp]`.

**Indice** : après unfolding automatique par `simp`, la somme sur `Finset.range 0` est vide et le slash par l'identité (cas $p = 0$ de `heckeDiagMatrix`) restitue $f$.

In [6]:
-- Exercice 2 : le cas dégénéré T₀ = id.
--
-- Objectif : établir, pour tout poids k et toute fonction f,
--   ModularForm.heckeT k 0 f = f
-- TODO étudiant : à compléter (une seule tactique suffit).

#check @ModularForm.heckeT_zero_left   -- le lemme disponible

-- Exercice 2 : le cas dégénéré T₀ = id.
--
-- Objectif : établir, pour tout poids k et toute fonction f,
--   ModularForm.heckeT k 0 f = f
-- TODO étudiant : à compléter (une seule tactique suffit).

#check @ModularForm.heckeT_zero_left   -- le lemme disponible
──────▶  heckeT_zero_left : ∀ (k : ℤ) (f : UpperHalfPlane → ℂ), heckeT k 0 f = f
--% env 5

Raw input:
{"cmd": "-- Exercice 2 : le cas d\u00e9g\u00e9n\u00e9r\u00e9 T\u2080 = id.\n--\n-- Objectif : \u00e9tablir, pour tout poids k et toute fonction f,\n--   ModularForm.heckeT k 0 f = f\n-- TODO \u00e9tudiant : \u00e0 compl\u00e9ter (une seule tactique suffit).\n\n#check @ModularForm.heckeT_zero_left   -- le lemme disponible", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "heckeT_zero_left : ∀ (k : ℤ) (f : UpperHalfPlane → ℂ), heckeT k 0 f = f"}],
 "env": 5}

### Linéarité : des opérateurs, pas des transformations

La section d'après regroupe les théorèmes de linéarité — $T_p$ et $U_p$ commutent avec l'**addition** (`heckeT_add`, `heckeU_add`), la **multiplication scalaire** (`heckeT_smul`, `heckeU_smul`) et par conséquent la soustraction (`heckeT_sub`) et le passage à l'opposé (`heckeT_neg`). Techniquement, ces énoncés sont des conséquences de la structure du slash (`SlashAction.add_slash`, `smul_slash`) et de la linéarité des sommes finies — mais le lake les énonce et les prouve **à la main** pour chacun des deux opérateurs.

C'est ce qui autorise, en théorie des formes modulaires, la question spectrale : sur l'espace (de dimension finie) des formes modulaires de poids $k$, les $T_p$ commutent entre eux et diagonalisent simultanément — les **valeurs propres** $\lambda_p$ relient alors l'analyse (opérateurs) et l'arithmétique (coefficients, section 5).

In [7]:
-- La linéarité de T_p et U_p en l'argument :
#check @ModularForm.heckeT_add
#check @ModularForm.heckeU_add
#check @ModularForm.heckeT_smul
#check @ModularForm.heckeU_smul
#check @ModularForm.heckeT_neg
#check @ModularForm.heckeT_sub

-- La linéarité de T_p et U_p en l'argument :
#check @ModularForm.heckeT_add
──────▶  heckeT_add : ∀ (k : ℤ) (p : ℕ) (f g : UpperHalfPlane → ℂ), heckeT k p (f + g) = heckeT k p f + heckeT k p g
#check @ModularForm.heckeU_add
──────▶  heckeU_add : ∀ (k : ℤ) (p : ℕ) (f g : UpperHalfPlane → ℂ), heckeU k p (f + g) = heckeU k p f + heckeU k p g
#check @ModularForm.heckeT_smul
──────▶  heckeT_smul : ∀ (k : ℤ) (p : ℕ) (c : ℂ) (f : UpperHalfPlane → ℂ), heckeT k p (c • f) = c • heckeT k p f
#check @ModularForm.heckeU_smul
──────▶  heckeU_smul : ∀ (k : ℤ) (p : ℕ) (c : ℂ) (f : UpperHalfPlane → ℂ), heckeU k p (c • f) = c • heckeU k p f
#check @ModularForm.heckeT_neg
──────▶  heckeT_neg : ∀ (k : ℤ) (p : ℕ) (f : UpperHalfPlane → ℂ), heckeT k p (-f) = -heckeT k p f
#check @ModularForm.heckeT_sub
──────▶  heckeT_sub : ∀ (k : ℤ) (p : ℕ) (f g : UpperHalfPlane → ℂ), heckeT k p (f - g) = heckeT k p f - heckeT k p g
--% env 6

Raw input:
{"cmd": "-- La lin\u00e9arit\u00e9 de T_p et U_p en l'argument :\n#check @ModularForm.heckeT_add\n#check @ModularForm.heckeU_add\n#check @ModularForm.heckeT_smul\n#check @ModularForm.heckeU_smul\n#check @ModularForm.heckeT_neg\n#check @ModularForm.heckeT_sub", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "heckeT_add : ∀ (k : ℤ) (p : ℕ) (f g : UpperHalfPlane → ℂ), heckeT k p (f + g) = heckeT k p f + heckeT k p g"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "heckeU_add : ∀ (k : ℤ) (p : ℕ) (f g : UpperHalfPlane → ℂ), heckeU k p (f + g) = heckeU k p f + heckeU k p g"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "heckeT_smul : ∀ (k : ℤ) (p : ℕ) (c : ℂ) (f : UpperHalfPlane → ℂ), heckeT k p (c • f) = c • heckeT k p f"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "heckeU_smul : ∀ (k : ℤ) (p : ℕ) (c : ℂ) (f : UpperHalfPlane → ℂ), heckeU k p (c • f) = c • heckeU k p f"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "heckeT_neg : ∀ (k : ℤ) (p : ℕ) (f : UpperHalfPlane → ℂ), heckeT k p (-f) = -heckeT k p f"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "heckeT_sub : ∀ (k : ℤ) (p : ℕ) (f g : UpperHalfPlane → ℂ), heckeT k p (f - g) = heckeT k p f - heckeT k p g"}],
 "env": 6}

### Lecture : la structure d'endomorphisme

**Sortie obtenue** : six égalités fonctionnelles — pour $T_p$ et $U_p$, la compatibilité avec `+`, `•`, `-` (unaire et binaire).

| Théorème | Énoncé informel |
|----------|-----------------|
| `heckeT_add` / `heckeU_add` | $T_p(f+g) = T_p f + T_p g$ |
| `heckeT_smul` / `heckeU_smul` | $T_p(c \cdot f) = c \cdot T_p f$ |
| `heckeT_neg`, `heckeT_sub` | conséquences : opposé et différence |

**Points clés** :

1. La preuve de `heckeT_add` mélange `heckeU_add` et `SlashAction.add_slash` puis referme par `abel` — un exemple représentatif du style du module : réutiliser les briques Mathlib, ne jamais re-démontrer la structure ;
2. Ces énoncés portent sur des fonctions **arbitraires** : aucune régularité ni modularité — la restriction aux formes modulaires (stabilité de $T_p$) est un théorème plus profond, hors périmètre du lake ;
3. `heckeU_smul` est prouvé par `simp` sur la définition — contraste utile avec `heckeT_smul` (un `rw` explicite) : même énoncé, poids de preuve différent selon l'opérateur.

## 5. La formule des coefficients : la combinatoire de $T_p$

Si $f(\tau) = \sum_{n \geq 0} a(n)\, q^n$ (avec $q = e^{2\pi i \tau}$), l'action de $T_p$ sur la suite $a$ se lit **termes à termes** — c'est le théorème combinatoire du module :

$$\big(T_p f\big)_n = a(np) + \begin{cases} p^{k-1}\, a(n/p) & \text{si } p \mid n \\ 0 & \text{sinon} \end{cases}$$

Le lake encode cette formule par une **définition** (`coeffHeckeT`) et deux lectures conditionnelles démontrées (`coeffHeckeT_of_dvd`, `coeffHeckeT_of_not_dvd`). La partie $U$ correspond à l'échantillonnage pur `coeffHeckeU p a n = a (n * p)` — un simple « prélèvement » d'un coefficient sur $p$, sans aucun facteur.

La section `Examples` du lake (absente du dépôt amont) exécute cette formule sur la suite $a(n) = n$ au poids $k = 12$ — le poids de la forme modulaire **discriminant** $\Delta$. Les cellules qui suivent les **reproduisent en-kernel** puis les étendent.

In [8]:
-- La formule des coefficients et ses lectures conditionnelles :
#check @ModularForm.coeffHeckeT_apply
#check @ModularForm.coeffHeckeT_of_dvd      -- p ∣ n : les DEUX termes
#check @ModularForm.coeffHeckeT_of_not_dvd  -- p ∤ n : échantillonnage seul
#check @ModularForm.coeffHeckeU_apply

-- Exemples calculables (section Examples du lake), pour a n = n et k = 12 :
example : coeffHeckeU 2 (fun n => (n : ℂ)) 3 = 6 := rfl

example : coeffHeckeT 12 2 (fun n => (n : ℂ)) 1 = 2 := by
  have h : ¬ (2 : ℕ) ∣ 1 := by decide
  simp only [coeffHeckeT, if_neg h]
  norm_num

example : coeffHeckeT 12 2 (fun n => (n : ℂ)) 2 = 4 + 2 ^ 11 := by
  have h : (2 : ℕ) ∣ 2 := by decide
  simp only [coeffHeckeT, if_pos h]
  norm_num

example : coeffHeckeT 12 3 (fun n => (n : ℂ)) 3 = 9 + 3 ^ 11 := by
  have h : (3 : ℕ) ∣ 3 := by decide
  simp only [coeffHeckeT, if_pos h]
  norm_num

-- La formule des coefficients et ses lectures conditionnelles :
#check @ModularForm.coeffHeckeT_apply
──────▶  coeffHeckeT_apply : ∀ (k : ℤ) (p : ℕ) (a : ℕ → ℂ) (n : ℕ),
  coeffHeckeT k p a n = a (n * p) + if p ∣ n then ↑p ^ (k - 1) * a (n / p) else 0
#check @ModularForm.coeffHeckeT_of_dvd      -- p ∣ n : les DEUX termes
──────▶  coeffHeckeT_of_dvd : ∀ (k : ℤ) {p n : ℕ},
  p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p) + ↑p ^ (k - 1) * a (n / p)
#check @ModularForm.coeffHeckeT_of_not_dvd  -- p ∤ n : échantillonnage seul
──────▶  coeffHeckeT_of_not_dvd : ∀ (k : ℤ) {p n : ℕ}, ¬p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p)
#check @ModularForm.coeffHeckeU_apply
──────▶  coeffHeckeU_apply : ∀ (p : ℕ) (a : ℕ → ℂ) (n : ℕ), coeffHeckeU p a n = a (n * p)

-- Exemples calculables (section Examples du lake), pour a n = n et k = 12 :
example : coeffHeckeU 2 (fun n => (n : ℂ)) 3 = 6 := rfl

example : coeffHeckeT 12 2 (fun n => (n : ℂ)) 1 = 2 := by
  have h : ¬ (2 : ℕ) ∣ 1 := by decide
  simp only [coeffHeckeT, if_neg h]
  norm_num

example : coeffHeckeT 12 2 (fun n => (n : ℂ)) 2 = 4 + 2 ^ 11 := by
  have h : (2 : ℕ) ∣ 2 := by decide
  simp only [coeffHeckeT, if_pos h]
  norm_num

example : coeffHeckeT 12 3 (fun n => (n : ℂ)) 3 = 9 + 3 ^ 11 := by
  have h : (3 : ℕ) ∣ 3 := by decide
  simp only [coeffHeckeT, if_pos h]
  norm_num
--% env 7

Raw input:
{"cmd": "-- La formule des coefficients et ses lectures conditionnelles :\n#check @ModularForm.coeffHeckeT_apply\n#check @ModularForm.coeffHeckeT_of_dvd      -- p \u2223 n : les DEUX termes\n#check @ModularForm.coeffHeckeT_of_not_dvd  -- p \u2224 n : \u00e9chantillonnage seul\n#check @ModularForm.coeffHeckeU_apply\n\n-- Exemples calculables (section Examples du lake), pour a n = n et k = 12 :\nexample : coeffHeckeU 2 (fun n => (n : \u2102)) 3 = 6 := rfl\n\nexample : coeffHeckeT 12 2 (fun n => (n : \u2102)) 1 = 2 := by\n  have h : \u00ac (2 : \u2115) \u2223 1 := by decide\n  simp only [coeffHeckeT, if_neg h]\n  norm_num\n\nexample : coeffHeckeT 12 2 (fun n => (n : \u2102)) 2 = 4 + 2 ^ 11 := by\n  have h : (2 : \u2115) \u2223 2 := by decide\n  simp only [coeffHeckeT, if_pos h]\n  norm_num\n\nexample : coeffHeckeT 12 3 (fun n => (n : \u2102)) 3 = 9 + 3 ^ 11 := by\n  have h : (3 : \u2115) \u2223 3 := by decide\n  simp only [coeffHeckeT, if_pos h]\n  norm_num", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "coeffHeckeT_apply : ∀ (k : ℤ) (p : ℕ) (a : ℕ → ℂ) (n : ℕ),\n  coeffHeckeT k p a n = a (n * p) + if p ∣ n then ↑p ^ (k - 1) * a (n / p) else 0"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "coeffHeckeT_of_dvd : ∀ (k : ℤ) {p n : ℕ},\n  p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p) + ↑p ^ (k - 1) * a (n / p)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "coeffHeckeT_of_not_dvd : ∀ (k : ℤ) {p n : ℕ}, ¬p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "coeffHeckeU_apply : ∀ (p : ℕ) (a : ℕ → ℂ) (n : ℕ), coeffHeckeU p a n = a (n * p)"}],
 "env": 7}

### Lecture : les quatre calculs

**Sortie obtenue** : les quatre signatures d'en-tête (`coeffHeckeT_apply`, `coeffHeckeT_of_dvd`, `coeffHeckeT_of_not_dvd`, `coeffHeckeU_apply`) — puis **rien** : chaque `example` est une égalité close que Lean valide en silence. Une cellule d'exemples qui échoue produit une erreur ; une cellule qui réussit ne produit que les sorties explicites de ses `#check`.

| Énoncé | Cas | Valeur |
|--------|-----|--------|
| `coeffHeckeU 2 a 3 = 6` | échantillonnage | $a(6) = 6$, `rfl` pur |
| `coeffHeckeT 12 2 a 1 = 2` | $2 \nmid 1$ | $a(2) = 2$ seul |
| `coeffHeckeT 12 2 a 2 = 4 + 2¹¹` | $2 \mid 2$ | $a(4) + 2^{11} a(1)$ |
| `coeffHeckeT 12 3 a 3 = 9 + 3¹¹` | $3 \mid 3$ | $a(9) + 3^{11} a(1)$ |

**Points clés** :

1. Le premier exemple se referme par **`rfl`** : l'échantillonnage est une égalité définitionnelle — le compilateur la voit par simple unfolding, sans tactique ;
2. Les autres suivent le schéma du lake : décider de la divisibilité (`decide`), réécrire la conditionnelle (`if_pos` / `if_neg`), refermer l'arithmétique (`norm_num`) ;
3. Le poids $k = 12$ fait apparaître $2^{11}$ et $3^{11}$ — au poids 12, le facteur diagonal domine largement le terme d'échantillonnage (cf. section suivante).

> **Note technique** : ces `example` sont anonymes — ils ne déclarent pas de nom dans l'environnement. Pour les **citer** (ou vérifier leurs axiomes), il faudrait les nommer `theorem` ; le lake a fait ce choix pour sa section Examples, ce notebook les garde jetables.

### L'ordre de grandeur du facteur diagonal

La formule des coefficients fait coexister deux termes d'échelles très différentes : l'échantillonnage $a(np)$ croît **linéairement** en $p$ (pour $a(n) = n$), quand le facteur diagonal $p^{k-1}$ croît **exponentiellement**. Au poids $k = 12$, la cellule suivante calcule les valeurs exactes de $p^{11}$ pour $p = 2, 3, 5$, et la valeur $1 + 2^{11}$ — la **valeur du second exemple divisible** du lake (branche $p \mid n$), prise sur la suite $a \equiv 1$ à $n = 2$ (le lake la démontre : `coeffHeckeT 12 2 (fun _ => 1) 2 = 1 + 2 ^ 11` ; la suite constante n'est pas une suite propre de $T_2$ au sens global — c'est la valeur de cet exemple, pas une valeur propre).

In [9]:
-- L'ordre de grandeur du facteur diagonal p^{k-1} au poids k = 12 :
#eval 2 ^ 11      -- facteur diagonal de T₂
#eval 3 ^ 11      -- facteur diagonal de T₃
#eval 5 ^ 11      -- facteur diagonal de T₅
#eval 1 + 2 ^ 11  -- second exemple divisible : suite a ≡ 1, n = 2 (branche 2 ∣ 2)

-- L'ordre de grandeur du facteur diagonal p^{k-1} au poids k = 12 :
#eval 2 ^ 11      -- facteur diagonal de T₂
─────▶  2048
#eval 3 ^ 11      -- facteur diagonal de T₃
─────▶  177147
#eval 5 ^ 11      -- facteur diagonal de T₅
─────▶  48828125
#eval 1 + 2 ^ 11  -- second exemple divisible : suite a ≡ 1, n = 2 (branche 2 ∣ 2)
─────▶  2049
--% env 8

Raw input:
{"cmd": "-- L'ordre de grandeur du facteur diagonal p^{k-1} au poids k = 12 :\n#eval 2 ^ 11      -- facteur diagonal de T\u2082\n#eval 3 ^ 11      -- facteur diagonal de T\u2083\n#eval 5 ^ 11      -- facteur diagonal de T\u2085\n#eval 1 + 2 ^ 11  -- second exemple divisible : suite a \u2261 1, n = 2 (branche 2 \u2223 2)", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "2048"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "177147"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "48828125"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "2049"}],
 "env": 8}

### Lecture : $p^{k-1}$ écrase l'échantillonnage

**Sortie obtenue** : quatre naturels calculés par le compilateur — `2048`, `177147`, `48828125`, `2049`.

| Grandeur | Valeur | Signification |
|----------|--------|---------------|
| $2^{11}$ | 2 048 | facteur diagonal de $T_2$ au poids 12 |
| $3^{11}$ | 177 147 | facteur diagonal de $T_3$ |
| $5^{11}$ | 48 828 125 | facteur diagonal de $T_5$ |
| $1 + 2^{11}$ | 2 049 | second exemple divisible : suite $a \equiv 1$, $n = 2$ (branche $2 \mid 2$) |

**Points clés** :

1. Pour la forme discriminant $\Delta(\tau) = \sum \tau(n)\, q^n$ (poids 12, normalisée : $a(1) = 1$), $\Delta$ est **forme propre** de $T_2$ avec $\lambda_2 = \tau(2) = -24$. La formule à $n = 2$ impose alors $\tau(4) + 2^{11} \cdot a(1) = \lambda_2\, a(2)$, c'est-à-dire $\tau(4) + 2048 = (-24)^2 = 576$, soit $\tau(4) = -1472$ — exactement la valeur de Ramanujan. Le pont « forme propre » n'est pas prouvé dans le lake, mais le calcul colle à la théorie ;
2. La croissance exponentielle de $p^{k-1}$ est la marque du **poids** : c'est elle qui empêche les coefficients de Hecke d'être bornés indépendamment de $k$ ;
3. Ces `#eval` sont des calculs **décidables** sur `ℕ` — à distinguer des `example` sur `ℂ` de la section précédente, refermés par preuve et non par évaluation ($\mathbb{C}$ est non calculable en kernel).

### Exercice 3 — $T_5$ au poids 12 sur un indice divisible

Étendez le calcul au cas $p = 5$, $n = 10$ (avec $5 \mid 10$) : l'énoncé attendu est $\texttt{coeffHeckeT}\ 12\ 5\ a\ 10 = 50 + 5^{11} \cdot 2$ — c'est-à-dire $a(10 \cdot 5) + 5^{11} a(10/5)$.

**Indice** : même schéma que les exemples de la section — `decide` pour $5 \mid 10$, `if_pos` pour choisir la branche, `norm_num` pour l'arithmétique.

In [10]:
-- Exercice 3 : T₅ au poids 12 sur la suite a n = n, pour n = 10 (5 ∣ 10).
--
-- Objectif : établir
--   ModularForm.coeffHeckeT 12 5 (fun n => (n : ℂ)) 10 = 50 + 5 ^ 11 * 2
-- TODO étudiant : à compléter (même schéma que les exemples de la section 5).

#check @ModularForm.coeffHeckeT_of_dvd   -- le lemme à invoquer

-- Exercice 3 : T₅ au poids 12 sur la suite a n = n, pour n = 10 (5 ∣ 10).
--
-- Objectif : établir
--   ModularForm.coeffHeckeT 12 5 (fun n => (n : ℂ)) 10 = 50 + 5 ^ 11 * 2
-- TODO étudiant : à compléter (même schéma que les exemples de la section 5).

#check @ModularForm.coeffHeckeT_of_dvd   -- le lemme à invoquer
──────▶  coeffHeckeT_of_dvd : ∀ (k : ℤ) {p n : ℕ},
  p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p) + ↑p ^ (k - 1) * a (n / p)
--% env 9

Raw input:
{"cmd": "-- Exercice 3 : T\u2085 au poids 12 sur la suite a n = n, pour n = 10 (5 \u2223 10).\n--\n-- Objectif : \u00e9tablir\n--   ModularForm.coeffHeckeT 12 5 (fun n => (n : \u2102)) 10 = 50 + 5 ^ 11 * 2\n-- TODO \u00e9tudiant : \u00e0 compl\u00e9ter (m\u00eame sch\u00e9ma que les exemples de la section 5).\n\n#check @ModularForm.coeffHeckeT_of_dvd   -- le lemme \u00e0 invoquer", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "coeffHeckeT_of_dvd : ∀ (k : ℤ) {p n : ℕ},\n  p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p) + ↑p ^ (k - 1) * a (n / p)"}],
 "env": 9}

## 6. Transparence axiomatique, i18n et limites

Le lake revendique une propriété forte : **zéro `sorry`** — propriété établie en amont par le scan du source et le build du lake, déjà validés. Ce notebook la **corrobore par échantillonnage** : chaque `#print axioms` exécuté au fil des sections certifie le théorème interrogé, et chacun est revenu avec la seule liste des axiomes standards du raisonnement classique (extensionnalité propositionnelle, choix, quotients — la liste exacte est dans les sorties). Un `sorry` — même transitif, caché dans une dépendance — apparaîtrait comme `sorryAx` dans la liste du théorème qui en dépend : son absence sur les théorèmes échantillonnés atteste que ces preuves-là sont closes.

**Convention i18n** : chaque module français a son miroir `HeckeOperator_en.lean` (docstrings anglaises, énoncés byte-identiques, namespaces distincts — EPIC #4980). Ce notebook visite le module FR ; les énoncés cités existent à l'identique côté EN.

**Limites du périmètre** (assumées par le lake, grain aval) :

- le **produit de Petersson** et l'orthogonalité des opérateurs de Hecke ne sont pas formalisés ;
- les **cusp forms** elles-mêmes (croissance à la pointe, développement en $q$-série) ne sont pas construites : le pont « $T_p f$ a pour coefficients `coeffHeckeT k p a` » reste un encodage parallèle, pas un théorème de passage ;
- les opérateurs agissent sur des fonctions arbitraires — la **stabilité** de l'espace des formes modulaires par $T_p$ (et la théorie spectrale qui en découle) n'est pas dans le module.

In [11]:
-- Transparence axiomatique finale : trois théorèmes représentatifs du lake.
#check @ModularForm.slash_heckeMatrix_apply   -- la brique analytique
#check @ModularForm.coeffHeckeT_of_dvd        -- la brique combinatoire
#check @ModularForm.heckeT_smul               -- la brique structurelle

#print axioms ModularForm.slash_heckeMatrix_apply
#print axioms ModularForm.coeffHeckeT_of_dvd
#print axioms ModularForm.heckeT_smul

-- Transparence axiomatique finale : trois théorèmes représentatifs du lake.
#check @ModularForm.slash_heckeMatrix_apply   -- la brique analytique
──────▶  slash_heckeMatrix_apply : ∀ (k : ℤ) {p : ℕ},
  p ≠ 0 →
    ∀ (j : ℕ) (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),
      (f ∣[k] heckeMatrix p j) τ = (↑p)⁻¹ * f (heckeMatrix p j • τ)
#check @ModularForm.coeffHeckeT_of_dvd        -- la brique combinatoire
──────▶  coeffHeckeT_of_dvd : ∀ (k : ℤ) {p n : ℕ},
  p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p) + ↑p ^ (k - 1) * a (n / p)
#check @ModularForm.heckeT_smul               -- la brique structurelle
──────▶  heckeT_smul : ∀ (k : ℤ) (p : ℕ) (c : ℂ) (f : UpperHalfPlane → ℂ), heckeT k p (c • f) = c • heckeT k p f

#print axioms ModularForm.slash_heckeMatrix_apply
──────▶  'ModularForm.slash_heckeMatrix_apply' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ModularForm.coeffHeckeT_of_dvd
──────▶  'ModularForm.coeffHeckeT_of_dvd' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms ModularForm.heckeT_smul
──────▶  'ModularForm.heckeT_smul' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 10

Raw input:
{"cmd": "-- Transparence axiomatique finale : trois th\u00e9or\u00e8mes repr\u00e9sentatifs du lake.\n#check @ModularForm.slash_heckeMatrix_apply   -- la brique analytique\n#check @ModularForm.coeffHeckeT_of_dvd        -- la brique combinatoire\n#check @ModularForm.heckeT_smul               -- la brique structurelle\n\n#print axioms ModularForm.slash_heckeMatrix_apply\n#print axioms ModularForm.coeffHeckeT_of_dvd\n#print axioms ModularForm.heckeT_smul", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "slash_heckeMatrix_apply : ∀ (k : ℤ) {p : ℕ},\n  p ≠ 0 →\n    ∀ (j : ℕ) (f : UpperHalfPlane → ℂ) (τ : UpperHalfPlane),\n      (f ∣[k] heckeMatrix p j) τ = (↑p)⁻¹ * f (heckeMatrix p j • τ)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "coeffHeckeT_of_dvd : ∀ (k : ℤ) {p n : ℕ},\n  p ∣ n → ∀ (a : ℕ → ℂ), coeffHeckeT k p a n = a (n * p) + ↑p ^ (k - 1) * a (n / p)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "heckeT_smul : ∀ (k : ℤ) (p : ℕ) (c : ℂ) (f : UpperHalfPlane → ℂ), heckeT k p (c • f) = c • heckeT k p f"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "'ModularForm.slash_heckeMatrix_apply' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "'ModularForm.coeffHeckeT_of_dvd' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "'ModularForm.heckeT_smul' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 10}

### Lecture : trois briques, trois certificats

**Sortie obtenue** : les signatures de trois théorèmes représentatifs — l'analyse (`slash_heckeMatrix_apply`), la combinatoire (`coeffHeckeT_of_dvd`), la structure (`heckeT_smul`) — et leurs listes d'axiomes, toutes identiques et **sans `sorryAx`**.

| Théorème | Famille | Axiomes |
|----------|---------|---------|
| `slash_heckeMatrix_apply` | analyse sur $\mathbb{H}$ | standards uniquement |
| `coeffHeckeT_of_dvd` | coefficients | standards uniquement |
| `heckeT_smul` | linéarité | standards uniquement |

**Point clé** : ces trois certificats cohérents couvrent des théorèmes de familles différentes et établissent que **ces preuves échantillonnées** ne dépendent ni d'un axiome exotique ni de `sorryAx`. Ils ne constituent pas un audit exhaustif du module ; l'absence globale de `sorry` est établie séparément par le scan du source du lake.

## Conclusion

Ce compagnon a fait exécuter par le compilateur les trois familles d'énoncés du lake `hecke_lean` : les représentants $\gamma_{p,j}$ et diagonaux (valeurs, déterminants, positivité), l'action de slash décomposée (homothéties écrasées contre dilatation), les opérateurs $U_p$ et $T_p$ (définitions, lecture ponctuelle, linéarité) et la formule des coefficients — avec ses exemples calculables reproduits en-kernel et étendus (exercice 3). Tous les `#print axioms` sont revenus identiques : ils **certifient les théorèmes échantillonnés** ; l'absence globale de `sorry` est celle du scan du source et du build du lake, déjà validés en amont.

### Ce que le notebook a montré

| Niveau | Symbole | Preuve vue |
|--------|---------|------------|
| Géométrie | `val_heckeMatrix`, `det_heckeMatrix` | équations de valeur, déterminant exact |
| Analyse | `slash_heckeMatrix_apply`, `heckeT_apply` | le slash calculé sur chaque représentant |
| Combinatoire | `coeffHeckeT_of_dvd/_of_not_dvd` | $a(np) + p^{k-1} a(n/p)$ |
| Structure | `heckeT_add`, `heckeT_smul` | endomorphismes de l'espace des fonctions |

### Trois takeaways

1. **Le `#check` est un typage** : il valide la signature du théorème contre les oleans du lake — ce que vous voyez est ce qui est prouvé ;
2. **Le `#print axioms` est un certificat (par théorème)** : l'absence de `sorryAx` est la vérification mécanique que **ce théorème-là** a une preuve close ;
3. **Le `rfl`/`norm_num` sur exemples est un oracle** : la formule des coefficients se **calcule** réellement (échantillonnage `rfl`, branches `decide` + `if_pos/if_neg` + `norm_num`) — la théorie ne reste pas symbolique.

### Pour aller plus loin

- **Grain aval du lake** : produit de Petersson, cusp forms, pont $q$-série (suivre le README du lake) ;
- **La source amont** : le dépôt `anthropics/fermats-last-theorem`, fichier `Definitions/Def_ModularForm_HeckeOperator.lean` — attribution Apache-2.0 complète dans `hecke_lean/NOTICE.md` ;
- **Références théoriques** : Diamond & Shurman, *A First Course in Modular Forms* (ch. 5) ; Apostol, *Modular Functions and Dirichlet Series in Number Theory* (ch. 6) ; pour la perspective computationnelle sur les coefficients, le cours de Don Zagier sur les formes modulaires et la fonction τ de Ramanujan.

### Exercices — progression conseillée

Sans dévoiler les preuves, voici le chemin de chacun :

1. **Exercice 1** (représentants) : décharger l'hypothèse $p \neq 0$ pour $p = 3$ par décision, invoquer l'équation de valeur, puis refermer l'égalité de littéraux — attention aux coercitions $\uparrow 2$ contre $2$, la réflexion doit être explicite ;
2. **Exercice 2** (cas dégénéré) : le lemme voulu est déjà enregistré pour la simplification automatique — une seule tactique referme le but ;
3. **Exercice 3** (coefficients) : trancher $5 \mid 10$ par décision, choisir la bonne branche de la conditionnelle, normaliser l'arithmétique — les exemples de la section 5 donnent le gabarit.